# Pratham-117M: GPT-1-inspired language model from scratch

This notebook builds an independently initialized decoder-only Transformer, trains its tokenizer on a legally sourced training split, runs a small end-to-end pretraining experiment, evaluates it, and exports/reloads its artifacts. It does **not** load pretrained language-model weights or call a model API.

## Run from VS Code using Google Colab
1. Install the official **Google Colab** extension in VS Code (and its **Jupyter** dependency if prompted), then open this notebook locally.
2. Click **Select Kernel** (top right) → **Colab** → choose **Auto Connect** or a runtime. Sign in to your Google account when prompted. This connects the local notebook editor to a remote Colab runtime; you do not need to upload this notebook to the Colab website.
3. If prompted to choose hardware, select a GPU runtime such as **T4**. Run the first setup cell and confirm that `device` reports `cuda`. Availability and quotas depend on your account/runtime.
4. Run cells from top to bottom. The default is a bounded smoke experiment. The 117M configuration is available for inspection/opt-in, not a promise that a free runtime can complete original-scale pretraining.

VS Code edits the local `.ipynb`, while its selected Colab kernel executes code on Google's remote machine. The notebook checks the active runtime and continues on CPU when possible. Colab sessions disconnect and GPU quotas vary; Drive checkpoints are used when mounted. If no GPU runtime is available, the smoke configuration still works on CPU but may run more slowly.

### Research integrity
The target architecture follows GPT-1's published dimensions (12 blocks, width 768, 12 heads, MLP 3072, context 512). This notebook's byte-level BPE, data source, software, and practical training loop are modern approximations—not an exact reproduction of OpenAI's original corpus, tokenizer, or compute budget. A smoke run is not GPT-1-equivalent.

Primary references: [GPT-1 paper](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf), [archived OpenAI code](https://github.com/openai/finetune-transformer-lm), and [Transformer paper](https://arxiv.org/abs/1706.03762).

## 1. Runtime, dependencies, and reproducibility

Colab supplies PyTorch and NumPy, so we do not replace its CUDA-enabled PyTorch build. We install only the tokenizer and safe tensor serialization packages at fixed versions. The checks below report Python, OS, PyTorch, CUDA, GPU memory, RAM, disk, and internet reachability.

In [ ]:
%pip -q install tokenizers==0.21.1 safetensors==0.5.3

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers 5.22.1 requires huggingface-hub<3.0,>=1.31.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers 5.22.1 requires safetensors>=0.8.0, but you have safetensors 0.5.3 which is incompatible.
transformers 5.22.1 requires tokenizers<0.24.0,>=0.23.1, but you have tokenizers 0.21.1 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires safetensors>=0.8.0, but you have safetensors 0.5.3 which is incompatible.


In [ ]:
import os, sys, re, json, math, time, random, hashlib, platform, urllib.request, urllib.error
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers
from safetensors.torch import save_file, load_file

SEED = 117
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print({'python': sys.version.split()[0], 'os': platform.platform(), 'torch': torch.__version__,
       'cuda_build': torch.version.cuda, 'device': str(device),
       'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'})
if torch.cuda.is_available():
    print('GPU memory GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
try:
    import psutil
    print('RAM GiB:', round(psutil.virtual_memory().total / 2**30, 2))
except ImportError:
    print('psutil not installed; RAM report skipped')
print('Python executable:', sys.executable)
try:
    urllib.request.urlopen('https://www.gutenberg.org', timeout=8).close()
    print('Internet: reachable')
except Exception as exc:
    print('Internet check failed:', type(exc).__name__, str(exc)[:180])
if not torch.cuda.is_available():
    print('No CUDA GPU selected. For T4, use Runtime > Change runtime type > T4 GPU. CPU smoke mode remains available.')

## 2. Persistent project storage

When running in Colab, mount Drive and store artifacts there so a runtime reset does not erase checkpoints. Outside Colab, use a local `Pratham/` folder. Drive mounting asks Google for authorization inside Colab; no credentials are embedded in this notebook.

In [ ]:
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/Pratham')
else:
    ROOT = Path.cwd() / 'Pratham'
for name in ['raw', 'processed', 'tokenizer', 'checkpoints', 'exports', 'logs']:
    (ROOT / name).mkdir(parents=True, exist_ok=True)
print('Project directory:', ROOT.resolve())

## 3. Public-domain text and provenance

BooksCorpus was used in GPT-1, but its original files are not a generally redistributable, legally straightforward source. This lab instead downloads plain-text Project Gutenberg works that are public domain in the United States. Check local jurisdiction and Project Gutenberg terms before reuse. Only a few books are downloaded for the smoke test; the manifest records identifiers, URLs, hashes, and the source. Downloads use retries and fail clearly rather than silently substituting data.

In [ ]:
BOOK_IDS = [1342, 84, 11]  # Pride and Prejudice, Frankenstein, Alice's Adventures in Wonderland
MAX_BOOKS = 3
def fetch_gutenberg(book_id, retries=3):
    url = f'https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt'
    target = ROOT / 'raw' / f'pg{book_id}.txt'
    if target.exists() and target.stat().st_size > 1000:
        return target, url
    last_error = None
    for attempt in range(retries):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Pratham-educational-research/1.0'})
            with urllib.request.urlopen(req, timeout=30) as response:
                data = response.read()
            if len(data) < 1000:
                raise ValueError(f'Unexpectedly short response for Gutenberg id {book_id}')
            target.write_bytes(data)
            return target, url
        except Exception as exc:
            last_error = exc
            time.sleep(1 + attempt)
    raise RuntimeError(f'Could not download Gutenberg id {book_id} after {retries} attempts: {last_error}')

def clean_text(text):
    text = text.replace('\ufeff', '').replace('\x00', ' ')
    text = re.sub(r'(?is)^.*?\*\*\*\s*START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*', '', text, count=1)
    text = re.sub(r'(?is)\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*$', '', text, count=1)
    text = re.sub(r'[\t\r ]+', ' ', text)
    text = re.sub(r'\n[ ]+', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

documents, manifest = [], []
for book_id in BOOK_IDS[:MAX_BOOKS]:
    path, url = fetch_gutenberg(book_id)
    raw = path.read_text(encoding='utf-8', errors='replace')
    text = clean_text(raw)
    digest = hashlib.sha256(text.encode('utf-8')).hexdigest()
    if len(text) >= 1000:
        documents.append({'id': str(book_id), 'text': text, 'sha256': digest})
        manifest.append({'document_id': str(book_id), 'source': 'Project Gutenberg', 'url': url,
                         'license_note': 'Public domain in the US; verify local law and source terms',
                         'cleaned_text_sha256': digest, 'characters': len(text)})
if len(documents) < 3:
    raise RuntimeError(f'Need at least 3 usable documents for document-level train/validation/test splits; got {len(documents)}')
raw_size = sum(len(d['text']) for d in documents)
print('documents:', len(documents), 'characters:', raw_size, 'approx words:', sum(len(d['text'].split()) for d in documents))
print('sample:', documents[0]['text'][:400])
(ROOT / 'processed' / 'dataset_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

### Document-level splitting before tokenizer fitting

Whole books are assigned to train/validation/test before the tokenizer sees any text. Duplicate normalized documents are removed first. With only three books, each split contains one book; validation/test estimates are consequently noisy and must not be interpreted as generalization benchmarks.

In [ ]:
seen, unique_docs = set(), []
for doc in documents:
    key = hashlib.sha256(re.sub(r'\s+', ' ', doc['text']).casefold().encode('utf-8')).hexdigest()
    if key not in seen:
        seen.add(key); unique_docs.append(doc)
if len(unique_docs) < 3:
    raise RuntimeError('Fewer than three unique books remain; add unique public-domain BOOK_IDS.')
order = list(range(len(unique_docs))); random.Random(SEED).shuffle(order)
n = len(order)
n_train, n_val = max(1, int(n * .8)), max(1, int(n * .1))
if n_train + n_val >= n: n_train, n_val = n - 2, 1
train_docs = [unique_docs[i] for i in order[:n_train]]
val_docs = [unique_docs[i] for i in order[n_train:n_train+n_val]]
test_docs = [unique_docs[i] for i in order[n_train+n_val:]]
split_ids = {'train': [d['id'] for d in train_docs], 'validation': [d['id'] for d in val_docs], 'test': [d['id'] for d in test_docs]}
assert not (set(split_ids['train']) & set(split_ids['validation']) or set(split_ids['train']) & set(split_ids['test']) or set(split_ids['validation']) & set(split_ids['test']))
(ROOT / 'processed' / 'split_ids.json').write_text(json.dumps(split_ids, indent=2), encoding='utf-8')
print('split ids:', split_ids)

## 4. Tokenization: a tiny BPE demonstration, then a trained tokenizer

Byte Pair Encoding starts with small symbols, counts adjacent pairs, and repeatedly merges the most frequent pair. This pedagogical implementation is intentionally tiny and operates on characters; production uses the optimized `tokenizers` byte-level BPE implementation and is fit on **training documents only**. Byte-level BPE is robust to arbitrary UTF-8 text, but does not imply multilingual language ability. It differs from GPT-1's historical text preprocessing/tokenizer.

In [ ]:
def tiny_bpe(words, merge_steps=5):
    corpus = [list(word) + ['</w>'] for word in words]
    merges = []
    for _ in range(merge_steps):
        counts = {}
        for tokens in corpus:
            for pair in zip(tokens, tokens[1:]): counts[pair] = counts.get(pair, 0) + 1
        if not counts: break
        best = max(counts, key=counts.get)
        merged = ''.join(best); merges.append((best, merged))
        updated = []
        for tokens in corpus:
            result, i = [], 0
            while i < len(tokens):
                if i + 1 < len(tokens) and (tokens[i], tokens[i+1]) == best:
                    result.append(merged); i += 2
                else:
                    result.append(tokens[i]); i += 1
            updated.append(result)
        corpus = updated
    return corpus, merges
demo_tokens, demo_merges = tiny_bpe(['low', 'lower', 'lowest', 'low'], 6)
print('merges:', demo_merges)
print('result:', demo_tokens)
assert len(demo_merges) > 0 and all(tokens for tokens in demo_tokens)

In [ ]:
SMOKE = {'name': 'smoke', 'vocab_target': 512, 'd_model': 192, 'n_layers': 4, 'n_heads': 4, 'd_ff': 768, 'context': 128, 'dropout': 0.1}
MINI = {'name': 'mini', 'vocab_target': 16000, 'd_model': 384, 'n_layers': 8, 'n_heads': 8, 'd_ff': 1536, 'context': 512, 'dropout': 0.1}
GPT1_INSPIRED = {'name': 'pratham-117m', 'vocab_target': 40478, 'd_model': 768, 'n_layers': 12, 'n_heads': 12, 'd_ff': 3072, 'context': 512, 'dropout': 0.1}
MODE = 'smoke'  # Change to 'mini' or 'pratham-117m' only after smoke verification.
cfg = {'smoke': SMOKE, 'mini': MINI, 'pratham-117m': GPT1_INSPIRED}[MODE].copy()

special_tokens = ['[UNK]', '[BOS]', '[EOS]', '[PAD]']
tokenizer = Tokenizer(models.BPE(unk_token='[UNK]'))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()
trainer = trainers.BpeTrainer(vocab_size=cfg['vocab_target'], min_frequency=2,
    special_tokens=special_tokens, initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
tokenizer.train_from_iterator((d['text'] for d in train_docs), trainer=trainer, length=len(train_docs))
cfg['vocab_size'] = tokenizer.get_vocab_size()
if MODE == 'pratham-117m' and cfg['vocab_size'] != cfg['vocab_target']:
    raise RuntimeError(f"This corpus produced {cfg['vocab_size']} tokenizer entries, not the required {cfg['vocab_target']}. Add a larger eligible training corpus before selecting the 117M mode.")
tokenizer_path = ROOT / 'tokenizer' / f"{MODE}.json"
tokenizer.save(str(tokenizer_path))
print('trained vocab:', cfg['vocab_size'], 'target:', cfg['vocab_target'], 'path:', tokenizer_path)
for sample in ['Once upon a time.', 'Numbers: 12345!', 'café — 東京']:
    enc = tokenizer.encode(sample)
    print(sample, '->', enc.ids[:20], '->', tokenizer.decode(enc.ids))
assert tokenizer.encode('Pratham').ids
assert tokenizer.get_vocab_size() >= len(special_tokens)

## 5. From-scratch decoder-only Transformer

For hidden states `X` shaped `[batch, time, width]`, causal self-attention is `softmax(Q K^T / sqrt(head_width) + mask) V`. The triangular mask prevents a position from attending to future tokens. Each block uses multi-head attention, a residual addition and LayerNorm, followed by a two-layer GELU MLP, another residual, and LayerNorm (post-normalization). The tanh GELU approximation is `0.5 x (1 + tanh(sqrt(2/pi) * (x + 0.044715 x^3)))`.

The token embedding is tied to the output classifier. The target model uses learned absolute positions and no pretrained weights. The smoke model preserves the same block design but reduces dimensions for a short demonstration.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg['d_model'] % cfg['n_heads'] == 0
        self.n_heads, self.head_dim = cfg['n_heads'], cfg['d_model'] // cfg['n_heads']
        self.qkv = nn.Linear(cfg['d_model'], 3 * cfg['d_model'])
        self.proj = nn.Linear(cfg['d_model'], cfg['d_model'])
        self.attn_drop = nn.Dropout(cfg['dropout'])
        self.resid_drop = nn.Dropout(cfg['dropout'])
    def forward(self, x, use_sdpa=True):
        b, t, c = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        def split(z): return z.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        q, k, v = split(q), split(k), split(v)
        if use_sdpa and hasattr(F, 'scaled_dot_product_attention'):
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.attn_drop.p if self.training else 0.0, is_causal=True)
        else:
            scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
            mask = torch.triu(torch.ones(t, t, device=x.device, dtype=torch.bool), diagonal=1)
            scores = scores.masked_fill(mask, torch.finfo(scores.dtype).min)
            weights = self.attn_drop(torch.softmax(scores, dim=-1))
            y = weights @ v
        y = y.transpose(1, 2).contiguous().view(b, t, c)
        return self.resid_drop(self.proj(y))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg['d_model']
        self.attn = CausalSelfAttention(cfg)
        self.ln1 = nn.LayerNorm(d, eps=1e-5)
        self.mlp = nn.Sequential(nn.Linear(d, cfg['d_ff']), nn.GELU(approximate='tanh'),
                                 nn.Linear(cfg['d_ff'], d), nn.Dropout(cfg['dropout']))
        self.ln2 = nn.LayerNorm(d, eps=1e-5)
    def forward(self, x):
        x = self.ln1(x + self.attn(x))
        return self.ln2(x + self.mlp(x))

class PrathamLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg.copy()
        d = cfg['d_model']
        self.token_emb = nn.Embedding(cfg['vocab_size'], d)
        self.pos_emb = nn.Embedding(cfg['context'], d)
        self.emb_drop = nn.Dropout(cfg['dropout'])
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg['n_layers'])])
        self.final_ln = nn.LayerNorm(d, eps=1e-5)
        self.lm_head = nn.Linear(d, cfg['vocab_size'], bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)
    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None: nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight); nn.init.zeros_(module.bias)
    def forward(self, idx, targets=None):
        if idx.ndim != 2 or idx.size(1) > self.cfg['context']:
            raise ValueError(f'Expected [batch,time] with time <= {self.cfg["context"]}, got {tuple(idx.shape)}')
        if idx.numel() and (idx.min() < 0 or idx.max() >= self.cfg['vocab_size']):
            raise ValueError('Input contains token IDs outside the configured vocabulary')
        pos = torch.arange(idx.size(1), device=idx.device)
        x = self.emb_drop(self.token_emb(idx) + self.pos_emb(pos)[None, :, :])
        for block in self.blocks: x = block(x)
        logits = self.lm_head(self.final_ln(x))
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1)) if targets is not None else None
        return logits, loss

model = PrathamLM(cfg).to(device)
parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('mode:', MODE, 'parameters:', f'{parameter_count:,}', 'FP32 weights MiB:', round(parameter_count * 4 / 2**20, 1))
print('weight tying:', model.lm_head.weight.data_ptr() == model.token_emb.weight.data_ptr())

### Structural and causal tests

A causal test perturbs only a future token and checks that earlier-position logits are unchanged in evaluation mode. The parameter total is computed from the actual instantiated model, accounting for tied weights only once by PyTorch's parameter iterator.

In [ ]:
model.eval()
x = torch.randint(0, cfg['vocab_size'], (2, min(12, cfg['context'])), device=device)
with torch.no_grad():
    logits_a, _ = model(x)
    x_changed = x.clone(); x_changed[:, -1] = (x_changed[:, -1] + 1) % cfg['vocab_size']
    logits_b, _ = model(x_changed)
assert logits_a.shape == (2, x.size(1), cfg['vocab_size'])
assert torch.allclose(logits_a[:, :-1], logits_b[:, :-1], atol=1e-5, rtol=1e-5), 'Future token leaked into earlier logits'
assert model.lm_head.weight.data_ptr() == model.token_emb.weight.data_ptr()
d, f, v, t, layers = 768, 3072, 40478, 512, 12
target_formula = v*d + t*d + layers*(4*d*d + 2*d*f + 9*d + f) + 2*d
assert target_formula == 116_536_320
attn = model.blocks[0].attn
z = torch.randn(1, 5, cfg['d_model'], device=device)
q, k, v = attn.qkv(z).chunk(3, dim=-1)
def split_heads(qkv): return qkv.view(1, 5, attn.n_heads, attn.head_dim).transpose(1, 2)
q, k, v = split_heads(q), split_heads(k), split_heads(v)
manual_scores = (q @ k.transpose(-2, -1)) / math.sqrt(attn.head_dim)
causal = torch.triu(torch.ones(5, 5, device=device, dtype=torch.bool), diagonal=1)
manual_scores = manual_scores.masked_fill(causal, torch.finfo(manual_scores.dtype).min)
manual_result = torch.softmax(manual_scores, dim=-1) @ v
sdpa_result = F.scaled_dot_product_attention(q, k, v, is_causal=True)
assert torch.allclose(manual_result, sdpa_result, atol=1e-5, rtol=1e-5), 'Manual and SDPA attention differ'
model.train()
test_x = torch.randint(0, cfg['vocab_size'], (2, 8), device=device)
test_y = torch.randint(0, cfg['vocab_size'], (2, 8), device=device)
_, test_loss = model(test_x, test_y); test_loss.backward()
assert torch.isfinite(test_loss) and model.token_emb.weight.grad is not None
print('PASS: shapes, causal masking, tied embeddings, finite loss, backward gradients')
model.zero_grad(set_to_none=True)

### Target-size estimate

For the tied-output implementation, instantiate the published-size configuration only when you have enough memory. A forward/backward pass at batch 1 and context 512 can still need several GiB for activations and optimizer state. An approximate training compute estimate is `6 × parameter_count × processed_tokens`; it omits attention's sequence-length-dependent cost and implementation overhead.

In [ ]:
def count_parameters(config):
    probe = dict(config); probe['vocab_size'] = config['vocab_target']
    temp = PrathamLM(probe)
    return sum(p.numel() for p in temp.parameters() if p.requires_grad)
target_params = count_parameters(GPT1_INSPIRED)
print('Actual parameter count at target dimensions:', f'{target_params:,}')
print('FP32 parameter storage MiB:', round(target_params * 4 / 2**20, 1))
print('FP16 parameter storage MiB:', round(target_params * 2 / 2**20, 1))
print('Lower-bound Adam training state (weights + grads + two FP32 moments) GiB:', round(target_params * 16 / 2**30, 2))
del temp

## 6. Next-token data and training objective

For each token window `[x0, x1, ..., xT]`, inputs are `[x0, ..., x(T-1)]` and targets are `[x1, ..., xT]`. The model minimizes mean cross-entropy over next-token logits. Perplexity is `exp(mean token loss)`; compare it only when tokenizer and test corpus match. This stream packing places an EOS token between books and keeps document splits separate.

In [ ]:
eos_id = tokenizer.token_to_id('[EOS]')
def encode_split(docs):
    ids = []
    for doc in docs:
        ids.extend(tokenizer.encode(doc['text']).ids)
        ids.append(eos_id)
    return torch.tensor(ids, dtype=torch.long)
train_ids, val_ids, test_ids = map(encode_split, (train_docs, val_docs, test_docs))
class WindowDataset(Dataset):
    def __init__(self, token_ids, context):
        self.ids, self.context = token_ids, context
        self.count = max(0, (len(token_ids) - 1) // context)
    def __len__(self): return self.count
    def __getitem__(self, i):
        start = i * self.context
        chunk = self.ids[start:start + self.context + 1]
        return chunk[:-1], chunk[1:]
train_ds = WindowDataset(train_ids, cfg['context'])
val_ds = WindowDataset(val_ids, cfg['context'])
test_ds = WindowDataset(test_ids, cfg['context'])
if len(train_ds) == 0 or len(val_ds) == 0 or len(test_ds) == 0:
    raise RuntimeError('A split is shorter than one context window; use a smaller context or larger legal corpus.')
MICROBATCH = 1 if MODE == 'pratham-117m' else 4
train_loader = DataLoader(train_ds, batch_size=MICROBATCH, shuffle=True, drop_last=True, pin_memory=torch.cuda.is_available())
xb, yb = next(iter(train_loader))
assert torch.equal(xb[:, 1:], yb[:, :-1])
print('batch:', tuple(xb.shape), 'train/val/test windows:', len(train_ds), len(val_ds), len(test_ds))
print('decoded sample:', tokenizer.decode(xb[0, :80].tolist()))

## 7. Bounded smoke pretraining with resume support

GPT-1 reported Adam, peak learning rate `2.5e-4`, warmup, and cosine decay. We use AdamW as a documented modern approximation and apply weight decay only to non-embedding matrix weights (not biases, embeddings, or LayerNorm). Default training is deliberately short. Increase steps only after checking GPU memory, corpus size, and runtime limits. Checkpoints are written atomically and carry config and tokenizer hashes.

In [ ]:
TRAIN_STEPS = 60
GRAD_ACCUM = 64 if MODE == 'pratham-117m' else 2
PEAK_LR = 2.5e-4
WARMUP_STEPS = min(10, max(1, TRAIN_STEPS // 5))
WEIGHT_DECAY = 0.01
tokenizer_hash = hashlib.sha256(tokenizer_path.read_bytes()).hexdigest()
config_hash = hashlib.sha256(json.dumps(cfg, sort_keys=True).encode()).hexdigest()
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + MODE
decay, no_decay = [], []
for name, param in model.named_parameters():
    if not param.requires_grad: continue
    (decay if param.ndim >= 2 and 'emb' not in name else no_decay).append(param)
optimizer = torch.optim.AdamW([{'params': decay, 'weight_decay': WEIGHT_DECAY},
                               {'params': no_decay, 'weight_decay': 0.0}], lr=PEAK_LR, betas=(0.9, 0.999))
def learning_rate(step):
    if step < WARMUP_STEPS: return PEAK_LR * (step + 1) / WARMUP_STEPS
    progress = min(1.0, (step - WARMUP_STEPS) / max(1, TRAIN_STEPS - WARMUP_STEPS))
    return PEAK_LR * 0.5 * (1 + math.cos(math.pi * progress))
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
metrics = {'run_id': run_id, 'started_utc': datetime.now(timezone.utc).isoformat(),
           'seed': SEED, 'config': cfg, 'config_sha256': config_hash,
           'tokenizer_sha256': tokenizer_hash, 'train_loss': [], 'validation_loss': [], 'lr': []}
checkpoint_path = ROOT / 'checkpoints' / f'{MODE}-latest.pt'
def save_checkpoint(path, state):
    tmp = path.with_suffix(path.suffix + '.tmp')
    torch.save(state, tmp)
    os.replace(tmp, path)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    path.with_suffix(path.suffix + '.sha256').write_text(digest, encoding='ascii')
def load_checkpoint(path):
    expected = path.with_suffix(path.suffix + '.sha256').read_text(encoding='ascii').strip()
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    if expected != actual: raise RuntimeError('Checkpoint integrity hash mismatch; use the previous valid checkpoint.')
    state = torch.load(path, map_location=device, weights_only=False)
    if state['config_sha256'] != config_hash or state['tokenizer_sha256'] != tokenizer_hash:
        raise RuntimeError('Checkpoint config/tokenizer differs from this run; refusing unsafe resume.')
    return state
print('run:', run_id, 'steps:', TRAIN_STEPS, 'microbatch:', MICROBATCH, 'grad accumulation:', GRAD_ACCUM, 'device:', device)

In [ ]:
start_step = 0
if checkpoint_path.exists() and checkpoint_path.with_suffix(checkpoint_path.suffix + '.sha256').exists():
    state = load_checkpoint(checkpoint_path)
    model.load_state_dict(state['model']); optimizer.load_state_dict(state['optimizer'])
    scaler.load_state_dict(state['scaler']); start_step = state['step']
    metrics = state.get('metrics', metrics)
    run_id = metrics.get('run_id', run_id)
    if 'torch_rng' in state: torch.set_rng_state(state['torch_rng'])
    if 'python_rng' in state: random.setstate(state['python_rng'])
    print('Resuming at step', start_step)

initial_token_weights = model.token_emb.weight.detach().clone()
train_iter = iter(train_loader)
model.train(); optimizer.zero_grad(set_to_none=True)
t0 = time.time()
for step in range(start_step, TRAIN_STEPS):
    lr = learning_rate(step)
    for group in optimizer.param_groups: group['lr'] = lr
    running_loss = 0.0
    for _ in range(GRAD_ACCUM):
        try: xb, yb = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader); xb, yb = next(train_iter)
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        amp_enabled = device.type == 'cuda'
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
            _, loss = model(xb, yb)
            scaled_loss = loss / GRAD_ACCUM
        if not torch.isfinite(loss): raise FloatingPointError(f'Non-finite training loss at step {step}')
        scaler.scale(scaled_loss).backward()
        running_loss += loss.detach().float().item() / GRAD_ACCUM
    scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    if not torch.isfinite(grad_norm): raise FloatingPointError(f'Non-finite gradient norm at step {step}')
    scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
    metrics['train_loss'].append(running_loss); metrics['lr'].append(lr)
    if (step + 1) % 10 == 0 or step == start_step:
        model.eval(); vals = []
        with torch.no_grad():
            for vx, vy in DataLoader(val_ds, batch_size=2):
                vx, vy = vx.to(device), vy.to(device)
                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == 'cuda')):
                    _, vl = model(vx, vy)
                vals.append(float(vl.float().item()))
        val_loss = float(np.mean(vals)); metrics['validation_loss'].append({'step': step + 1, 'loss': val_loss})
        print(f'step {step+1:>4}/{TRAIN_STEPS} train_loss={running_loss:.4f} val_loss={val_loss:.4f} ppl={math.exp(min(val_loss, 20)):.2f} lr={lr:.2e} grad={float(grad_norm):.3f}')
        model.train()
    if (step + 1) % 20 == 0 or step + 1 == TRAIN_STEPS:
        save_checkpoint(checkpoint_path, {'step': step + 1, 'model': model.state_dict(),
            'optimizer': optimizer.state_dict(), 'scaler': scaler.state_dict(),
            'config_sha256': config_hash, 'tokenizer_sha256': tokenizer_hash,
            'metrics': metrics, 'torch_rng': torch.get_rng_state(), 'python_rng': random.getstate()})
metrics['elapsed_seconds'] = time.time() - t0
metrics['tokens_processed_estimate'] = len(metrics['train_loss']) * GRAD_ACCUM * MICROBATCH * cfg['context']
metrics['finished_utc'] = datetime.now(timezone.utc).isoformat()
if start_step < TRAIN_STEPS:
    assert not torch.equal(initial_token_weights, model.token_emb.weight.detach()), 'Training did not change model weights'
(ROOT / 'logs' / f'{run_id}-metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print('elapsed seconds:', round(metrics['elapsed_seconds'], 1), 'estimated tokens:', metrics['tokens_processed_estimate'])

## 8. Held-out evaluation and text generation

Test loss is computed on book(s) excluded from tokenizer training and model updates. Generation is implemented directly by repeatedly evaluating the model's final-position logits; strategies include temperature, top-k, nucleus (top-p), and repetition penalty. Randomly initialized/smoke-trained outputs are expected to be weak and are not edited or presented as capability claims.

In [ ]:
model.eval()
test_losses = []
with torch.no_grad():
    for tx, ty in DataLoader(test_ds, batch_size=2):
        tx, ty = tx.to(device), ty.to(device)
        _, tl = model(tx, ty)
        test_losses.append(float(tl.item()))
test_loss = float(np.mean(test_losses))
evaluation = {'test_loss': test_loss, 'test_perplexity': math.exp(test_loss),
              'test_windows': len(test_ds), 'next_token_accuracy': None,
              'note': 'Small corpus/split; diagnostic only, not a benchmark.'}
print(evaluation)

@torch.no_grad()
def generate(prompt, max_new_tokens=80, temperature=0.8, top_k=40, top_p=0.9, repetition_penalty=1.1, seed=117):
    if temperature <= 0: raise ValueError('temperature must be > 0')
    generator = torch.Generator(device=device); generator.manual_seed(seed)
    ids = tokenizer.encode(prompt).ids
    if not ids: ids = [tokenizer.token_to_id('[BOS]')]
    out = torch.tensor([ids], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        ctx = out[:, -cfg['context']:]
        logits, _ = model(ctx); scores = logits[:, -1, :].float()
        if repetition_penalty > 1:
            for token_id in set(out[0].tolist()):
                scores[0, token_id] = scores[0, token_id] / repetition_penalty if scores[0, token_id] > 0 else scores[0, token_id] * repetition_penalty
        scores = scores / temperature
        if top_k and top_k > 0:
            cutoff = torch.topk(scores, min(top_k, scores.size(-1))).values[:, -1, None]
            scores = scores.masked_fill(scores < cutoff, -float('inf'))
        if top_p < 1:
            sorted_scores, sorted_idx = torch.sort(scores, descending=True)
            probs = torch.softmax(sorted_scores, dim=-1)
            remove = torch.cumsum(probs, dim=-1) - probs >= top_p
            sorted_scores[remove] = -float('inf')
            scores = torch.full_like(scores, -float('inf')).scatter(1, sorted_idx, sorted_scores)
        probs = torch.softmax(scores, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1, generator=generator)
        out = torch.cat([out, next_id], dim=1)
        if next_id.item() == tokenizer.token_to_id('[EOS]'): break
    return tokenizer.decode(out[0].tolist())

samples = {}
for prompt in ['Once upon a time', 'The history of science']:
    samples[prompt] = generate(prompt)
    print('\nPROMPT:', prompt, '\nOUTPUT:', samples[prompt])

## 9. Export, reload, and persist actual run outputs

A resumable training checkpoint contains optimizer/scaler state; exported safetensors contain only inference weights. Reload is verified in a fresh model instance by comparing logits. This cell writes only the current model's actual artifacts and a truthful status note. It does not publish or upload anything.

In [ ]:
export_dir = ROOT / 'exports' / MODE
export_dir.mkdir(parents=True, exist_ok=True)
model_path = export_dir / 'model.safetensors'
export_state = {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items() if k != 'lm_head.weight'}
save_file(export_state, str(model_path))
tokenizer.save(str(export_dir / 'tokenizer.json'))
(export_dir / 'config.json').write_text(json.dumps(cfg, indent=2), encoding='utf-8')
(export_dir / 'evaluation_results.json').write_text(json.dumps(evaluation, indent=2), encoding='utf-8')
(export_dir / 'training_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
(export_dir / 'dataset_manifest.json').write_text(json.dumps({'sources': manifest, 'splits': split_ids}, indent=2), encoding='utf-8')
fresh = PrathamLM(cfg).to(device)
reloaded_weights = load_file(str(model_path), device=str(device))
reloaded_weights['lm_head.weight'] = reloaded_weights['token_emb.weight']
fresh.load_state_dict(reloaded_weights)
fresh.eval()
probe = torch.randint(0, cfg['vocab_size'], (1, min(16, cfg['context'])), device=device)
with torch.no_grad():
    expected, _ = model(probe); actual, _ = fresh(probe)
assert torch.allclose(expected, actual, atol=1e-5, rtol=1e-5)
(export_dir / 'MODEL_CARD.md').write_text(
    f'# Pratham {MODE}\n\nRandomly initialized GPT-1-inspired decoder-only model.\n\n'
    f'Architecture config: `{json.dumps(cfg, sort_keys=True)}`.\n\n'
    'Training status and metrics are in training_metrics.json. This artifact is not an official OpenAI model, '
    'not GPT-1-equivalent, and may be undertrained. Dataset source/license notes are in dataset_manifest.json.\n', encoding='utf-8')
print('Fresh reload logits match. Exported to:', export_dir)

## 10. What this run does—and does not—establish

- **Verified if all cells pass:** data acquisition, deduplication/splits, tokenizer training on the train split, random initialization, causal masking, gradients, bounded optimization, held-out diagnostic loss, checkpoint integrity, generation, and fresh export/reload.
- **Not verified by a smoke run:** meaningful language capability, original BooksCorpus replication, original-scale 100-epoch training, GPT-1 benchmark reproduction, downstream task scores, or exact GPU bitwise determinism.
- **Full configuration:** the 117M model keeps 12 blocks, width 768, 12 heads, MLP 3072, 512 positions, and 40,478 target vocabulary. Building the tokenizer to that size requires a much larger eligible training corpus than this tiny demonstration. A larger vocabulary/model changes memory and compute substantially.
- **Resume:** rerun the setup/data/tokenizer/model cells with the same data, split seed, config and tokenizer; the training cell detects a matching latest checkpoint. A changed tokenizer/config is intentionally rejected.

The original paper's BooksCorpus and historical training setup are not reproduced here. The original OpenAI code is archived research code; this notebook uses modern PyTorch and byte-level BPE for practical compatibility. GPU kernels and runtime versions can also prevent exact bitwise reproducibility.